In [314]:
import pandas as pd
import numpy as np

In [315]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

In [316]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix

In [317]:
training_path = "../datasets/training_dataset.csv"
validation_path = "../datasets/validation_dataset.csv"

In [318]:
training_dataset = pd.read_csv(training_path)
validation_dataset = pd.read_csv(validation_path)

In [319]:
training_dataset.shape

(7000, 12)

In [320]:
x_train = training_dataset.iloc[:, :11]
y_train =  training_dataset.iloc[:, 11]

In [321]:
x_val = validation_dataset.iloc[:, :11]
y_val = validation_dataset.iloc[:, 11]

decision tree model

In [322]:
def metrics(y_test, pred):
    metrics = {}
    metrics["accuracy_score"] = accuracy_score(y_test, pred)
    metrics["precision_score"] = precision_score(y_test, pred)
    metrics["recall_score"] = recall_score(y_test, pred)
    metrics["f1_score"] = f1_score(y_test, pred)
    metrics["roc_auc_score"] = roc_auc_score(y_test, pred)
    metrics["confusion_matrix"] = confusion_matrix(y_test, pred)

    return metrics

In [323]:
report = {}

In [324]:
model1 = DecisionTreeClassifier(max_depth = 10)
model1.fit(x_train, y_train)
predict = model1.predict(x_val)
report["Decision_tree"] = metrics(y_val, predict)

In [325]:
model2 = RandomForestClassifier(max_depth=10)
model2.fit(x_train, y_train)
predict = model2.predict(x_val)
report["Random_Forest"] = metrics(y_val, predict)

In [326]:
model3 = SVC()
model3.fit(x_train, y_train)
predict = model3.predict(x_val)
report["SVC"] = metrics(y_val, predict)

In [327]:
model4 = KNeighborsClassifier()
model4.fit(x_train, y_train)
predict = model4.predict(x_val)
report["KNN"] = metrics(y_val, predict)

In [328]:
report["Decision_tree"]

{'accuracy_score': 0.8366666666666667,
 'precision_score': 0.5866666666666667,
 'recall_score': 0.4647887323943662,
 'f1_score': 0.518664047151277,
 'roc_auc_score': 0.6941542346182358,
 'confusion_matrix': array([[1123,   93],
        [ 152,  132]])}

In [329]:
report["Random_Forest"]

{'accuracy_score': 0.8753333333333333,
 'precision_score': 0.8012422360248447,
 'recall_score': 0.45422535211267606,
 'f1_score': 0.5797752808988764,
 'roc_auc_score': 0.7139547813194959,
 'confusion_matrix': array([[1184,   32],
        [ 155,  129]])}

In [330]:
report["KNN"]

{'accuracy_score': 0.8446666666666667,
 'precision_score': 0.625615763546798,
 'recall_score': 0.4471830985915493,
 'f1_score': 0.5215605749486653,
 'roc_auc_score': 0.6923415492957746,
 'confusion_matrix': array([[1140,   76],
        [ 157,  127]])}

In [331]:
report["SVC"]

{'accuracy_score': 0.8686666666666667,
 'precision_score': 0.8041958041958042,
 'recall_score': 0.40492957746478875,
 'f1_score': 0.5386416861826698,
 'roc_auc_score': 0.6909516308376575,
 'confusion_matrix': array([[1188,   28],
        [ 169,  115]])}

DecisionTree is our best bet, we will do hyperparameter finetunning for the same, since we are more focused on recall which indicates churn amount predicted correctly

In [332]:
second_iter_model = DecisionTreeClassifier(max_depth = 10)
second_iter_model.fit(x_train, y_train)
pred = second_iter_model.predict_proba(x_val)[:, 1]
pred = (pred >= 0.19).astype(int)
metrics(y_val, pred)

{'accuracy_score': 0.7726666666666666,
 'precision_score': 0.43537414965986393,
 'recall_score': 0.676056338028169,
 'f1_score': 0.5296551724137931,
 'roc_auc_score': 0.7356433005930318,
 'confusion_matrix': array([[967, 249],
        [ 92, 192]])}

In [333]:
from xgboost import XGBClassifier

model = XGBClassifier()
model.fit(x_train, y_train)
predict = model.predict(x_val)
report["xgboost"] = metrics(y_val, predict)

In [334]:
report["xgboost"]

{'accuracy_score': 0.8533333333333334,
 'precision_score': 0.6632653061224489,
 'recall_score': 0.45774647887323944,
 'f1_score': 0.5416666666666666,
 'roc_auc_score': 0.7017350815418829,
 'confusion_matrix': array([[1150,   66],
        [ 154,  130]])}

In [335]:
xgboost_final_model = XGBClassifier(max_depth = 7, learning_rate = 0.01,n_estimators = 1000)
xgboost_final_model.fit(x_train, y_train)
predict = xgboost_final_model.predict_proba(x_val)[:, 1]
predict = (predict >= 0.2).astype(int)
report["xgboost"] = metrics(y_val, predict)
report["xgboost"]

{'accuracy_score': 0.8053333333333333,
 'precision_score': 0.49099099099099097,
 'recall_score': 0.7676056338028169,
 'f1_score': 0.5989010989010989,
 'roc_auc_score': 0.7908751853224611,
 'confusion_matrix': array([[990, 226],
        [ 66, 218]])}

In [336]:
testing_dataset = pd.read_csv("../datasets/testing_dataset.csv")

In [337]:
x_test = testing_dataset.iloc[:, :11]
y_test =  testing_dataset.iloc[:, 11]

In [338]:
predict_prob = xgboost_final_model.predict_proba(x_test)[:, 1]
predict = (predict_prob >= 0.2).astype(int)
metrics(y_test, predict)

{'accuracy_score': 0.8046666666666666,
 'precision_score': 0.5079006772009029,
 'recall_score': 0.75,
 'f1_score': 0.6056527590847914,
 'roc_auc_score': 0.7841666666666667,
 'confusion_matrix': array([[982, 218],
        [ 75, 225]])}

In [339]:
import pickle

f = open("../model.pkl", "wb")
pickle.dump(xgboost_final_model, f)
f.close()